# VGG11: explicit per-layer mixed precision

Assign 8-bit or 4-bit signed activation/weight precision to named layers of a CIFAR-10 VGG11-BN checkpoint. Each layer owns an independent `Simulator`, its physical configuration, and programmed matrices. Set `layer_bits` to compare your assignment with uniform precision while varying ADC resolution.

Install MemFast, torchvision, NumPy, Matplotlib, and Jupyter. Set `MEMFAST_CIFAR10_ROOT` and `MEMFAST_VGG_CHECKPOINT` to local CIFAR-10 test data and `cifar10_vgg11_bn-eaeebf42.pt` from the pytorch-cifar-models VGG release. Default evaluation uses the first 512 test images; `MEMFAST_VGG_SAMPLES=0` evaluates the full test split. Choose CPU or CUDA in the settings cell.

In [ ]:
import copy
import gc
import os
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from memfast import Config, Nonidealities, Simulator
from memfast.models import cifar_model
from memfast.nn import AnalogConv2d, AnalogLinear

SEED = 135
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device(
    os.environ.get("MEMFAST_NOTEBOOK_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
)
data_root = Path(os.environ.get("MEMFAST_CIFAR10_ROOT", "data/cifar10"))
checkpoint = Path(
    os.environ.get(
        "MEMFAST_VGG_CHECKPOINT",
        str(Path(torch.hub.get_dir()) / "checkpoints/cifar10_vgg11_bn-eaeebf42.pt"),
    )
)
max_samples = int(os.environ.get("MEMFAST_VGG_SAMPLES", "512"))
batch_size = 8
if max_samples < 0:
    raise ValueError("Use a positive sample limit or 0 for the full test split")
if not checkpoint.is_file():
    raise FileNotFoundError(f"Set MEMFAST_VGG_CHECKPOINT to a real checkpoint: {checkpoint}")

## 1. Load and evaluate the original digital model

Use fixed ordered CIFAR-10 test inputs with the repository's evaluation normalization. Strict loading catches architecture or classifier mismatches. The table in the next cell lists the actual module paths used for layer selection; batch normalization, activations, pooling, and biases remain digital in all policies.

In [ ]:
preprocess = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)
dataset = datasets.CIFAR10(data_root, train=False, download=False, transform=preprocess)
sample_indices = list(range(len(dataset) if max_samples == 0 else min(max_samples, len(dataset))))
batches = list(DataLoader(Subset(dataset, sample_indices), batch_size=batch_size, shuffle=False))
targets = torch.cat([labels for _, labels in batches])
model = cifar_model("vgg11_bn", num_classes=10).float().eval()
model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True), strict=True)
layers = {
    name: layer
    for name, layer in model.named_modules()
    if isinstance(layer, (nn.Conv2d, nn.Linear))
}
weight_counts = {name: layer.weight.numel() for name, layer in layers.items()}
for name, layer in layers.items():
    print(f"{name:<16} {type(layer).__name__:<8} weight={tuple(layer.weight.shape)}")


@torch.inference_mode()
def evaluate(network):
    network.eval()
    logits = torch.cat([network(images.to(device)).cpu() for images, _ in batches])
    assert logits.shape == (len(targets), 10) and torch.isfinite(logits).all()
    predictions = logits.argmax(1)
    return dict(
        count=len(targets),
        correct=int((predictions == targets).sum()),
        top1=float((predictions == targets).float().mean()),
        cross_entropy=float(nn.functional.cross_entropy(logits, targets)),
        logits=logits,
    )


model.to(device)
digital_result = evaluate(model)
model.cpu()
print(
    f"Digital top-1: {100 * digital_result['top1']:.2f}% ({digital_result['correct']}/{len(targets)})"
)

## 2. Choose bits and ADC widths explicitly

Edit `layer_bits` below: every eligible module path must appear exactly once, with a value of 4 or 8. Here each value sets both activation bits and weight bits for that layer; the next layer independently requantizes its input. No global bit-width mutation or mixed-precision search is hidden in conversion.

Symmetric signed INT uses $q=\operatorname{clip}(\operatorname{round}(x/s),-Q,Q)$, $Q=2^{b-1}-1$, and $s=\max|x|/Q$. Thus reducing bits increases quantization steps. Quantization uses one scale per unfolded input vector/patch and one per output channel. Both policies keep physical arrays at 64 x 64 and binary slicing, independent of the chosen numerical precision.

`adc_bits` is the default converter width, independent of activation/weight bits. `adc_by_layer` overrides named layers and can contain either a scalar or a tuple with one entry per weight slice, **sign slice first**. With binary slicing the tuple length is 8 for an 8-bit layer or 4 for a 4-bit layer. For example, an 8-bit layer may use `(6, 7, 7, 8, 8, 8, 8, 8)`. The tuple is not the weight slice layout.

An ADC code is a rounded, clipped current count with maximum $2^{b_{ADC}}-1$. Sixty-four binary rows can produce count 64, so ADC6 can clip even with ideal devices, whereas ADC8 covers these ideal counts. Programming is $G'=G\exp(0.02z)$ by default; ideal-device comparisons explicitly set write variation to zero and retain quantization and ADC conversion.

In [ ]:
layer_bits = {
    "features.0": 8,
    "features.4": 8,
    "features.8": 4,
    "features.11": 4,
    "features.15": 4,
    "features.18": 4,
    "features.22": 4,
    "features.25": 4,
    "classifier.0": 4,
    "classifier.3": 4,
    "classifier.6": 8,
}
adc_bits = 8
adc_by_layer = {
    "features.0": 8,
    "classifier.6": 8,
}
physical_config = Config(
    array_size=(64, 64),
    adc_bits=adc_bits,
    seed=SEED,
    input_quant_gran=(1, None),
    weight_quant_gran=(None, 1),
    nonidealities=Nonidealities(write_variation=0.02),
)
if set(layer_bits) != set(layers) or any(value not in (4, 8) for value in layer_bits.values()):
    raise ValueError("Assign exactly every eligible layer either 4 or 8 bits")
if set(adc_by_layer) - set(layers):
    raise ValueError("ADC overrides contain unknown layer names")
for name in layers:
    print(
        f"{name:<16} A{layer_bits[name]} W{layer_bits[name]} ADC={adc_by_layer.get(name, adc_bits)}"
    )

## 3. Build one simulator per layer

Build each policy from a copy of the digital model. Replace layers by name to assign their simulators, precision and physical settings. The seed and original layer name determine the programmed fields for each layout.

The construction check verifies that simulators are distinct objects and each mapped matrix uses the intended layer configuration. Biases remain digital. The `logical_weight_bits` total is an arithmetic storage proxy, $\sum_l n_l b_l$, excluding biases and digital layers. It is not measured packed tensor memory, device area, latency, or energy.

In [ ]:
def build_mixed_model(bits_by_layer, default_adc, overrides, write=0.02):
    if set(bits_by_layer) != set(layers) or any(
        bits not in (4, 8) for bits in bits_by_layer.values()
    ):
        raise ValueError("Every eligible layer needs an explicit 4-bit or 8-bit assignment")
    if set(overrides) - set(layers):
        raise ValueError("Unknown layer in ADC overrides")
    converted = copy.deepcopy(model).to(device).eval()
    engines = {}
    for name, bits in bits_by_layer.items():
        source = converted.get_submodule(name)
        config = replace(
            physical_config,
            activation_bits=bits,
            weight_bits=bits,
            adc_bits=overrides.get(name, default_adc),
            nonidealities=Nonidealities(write_variation=write),
        )
        engine = Simulator(config, device=str(device))
        engines[name] = engine
        replacement = (
            AnalogConv2d(source, engine, name=name)
            if isinstance(source, nn.Conv2d)
            else AnalogLinear(source, engine, name=name)
        )
        parent_name, attribute = name.rsplit(".", 1)
        setattr(converted.get_submodule(parent_name), attribute, replacement)
        matrices = (
            list(replacement.matrices) if isinstance(replacement, AnalogConv2d) else [replacement]
        )
        assert all(matrix.simulator is engine and matrix.config == config for matrix in matrices)
    assert len({id(engine) for engine in engines.values()}) == len(layers)
    return converted.eval().requires_grad_(False), engines


user_model, user_engines = build_mixed_model(layer_bits, adc_bits, adc_by_layer)
user_result = evaluate(user_model)
user_configs = {name: asdict(engine.config) for name, engine in user_engines.items()}
print(f"User assignment: {100 * user_result['top1']:.2f}% top-1")
del user_model, user_engines
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

## 4. Compare a complete bounded policy/ADC/write grid

Declare five policies: uniform 8-bit, uniform 4-bit, the user-selected assignment, 8-bit first convolution/final head with other layers at 4-bit, and 8-bit feature extractor with a 4-bit classifier. Cross them with ADC {5, 6, 8} and write variation {0, 0.02}: **5 x 3 x 2 = 30 evaluations**, in addition to the digital baseline and the separate user-override demonstration.

The grid uses the indicated ADC width in every layer (`overrides={}`). The earlier user example uses `adc_by_layer` for per-layer ADC settings. Edit the policy, ADC and write-variation sets to choose the configurations to compare.

In [ ]:
policies = {
    "uniform8": {name: 8 for name in layers},
    "uniform4": {name: 4 for name in layers},
    "user": dict(layer_bits),
    "first-head8": {name: 8 if name in ("features.0", "classifier.6") else 4 for name in layers},
    "features8-head4": {name: 8 if name.startswith("features.") else 4 for name in layers},
}
adc_grid = (5, 6, 8)
write_grid = (0.0, 0.02)
print("Declared policy/ADC/write points:", len(policies) * len(adc_grid) * len(write_grid))
for label, assignment in policies.items():
    logical_bits = sum(weight_counts[name] * bits for name, bits in assignment.items())
    average = logical_bits / sum(weight_counts.values())
    print(
        f"{label:<18} mean weight bits={average:.3f}; 8-bit layers={[name for name, bits in assignment.items() if bits == 8]}"
    )

In [ ]:
mixed_results = []
for label, assignment in policies.items():
    for adc in adc_grid:
        for write in write_grid:
            started = time.perf_counter()
            analog, engines = build_mixed_model(assignment, adc, overrides={}, write=write)
            result = evaluate(analog)
            result.update(
                policy=label,
                adc_bits=adc,
                write_variation=write,
                assignment=dict(assignment),
                configs={name: asdict(engine.config) for name, engine in engines.items()},
                logical_weight_bits=sum(
                    weight_counts[name] * bits for name, bits in assignment.items()
                ),
                agreement=float(
                    (result["logits"].argmax(1) == digital_result["logits"].argmax(1))
                    .float()
                    .mean()
                ),
                seconds=time.perf_counter() - started,
            )
            mixed_results.append(result)
            print(
                f"{len(mixed_results):02d}/30 {label:<18} ADC{adc} write={write:g} top1={100 * result['top1']:.2f}%",
                flush=True,
            )
            del analog, engines
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
assert len(mixed_results) == len(policies) * len(adc_grid) * len(write_grid)
assert all(result["count"] == len(targets) for result in mixed_results)

## 5. Inspect the precision map and measured accuracy

The first panel shows the per-layer precision assignment. The other panels compare accuracy at each ADC width on the same vertical scale. Ideal-device CIM retains quantization and ADC conversion with programming variation set to zero.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), constrained_layout=True)
policy_names = list(policies)
precision_map = np.array([[policies[label][name] for label in policy_names] for name in layers])
image = axes[0].imshow(precision_map, vmin=4, vmax=8, cmap="cividis", aspect="auto")
axes[0].set_xticks(range(len(policy_names)), policy_names, rotation=45, ha="right")
axes[0].set_yticks(range(len(layers)), list(layers))
axes[0].set_title("Explicit activation/weight bits")
for row in range(len(layers)):
    for column in range(len(policy_names)):
        value = precision_map[row, column]
        axes[0].text(
            column,
            row,
            str(value),
            ha="center",
            va="center",
            color="black" if value == 8 else "white",
        )
fig.colorbar(image, ax=axes[0], ticks=[4, 8], label="Bits")
accuracy_values = [100 * item["top1"] for item in mixed_results] + [100 * digital_result["top1"]]
accuracy_limits = (max(0, min(accuracy_values) - 0.5), min(100, max(accuracy_values) + 0.5))
for ax, write in zip(axes[1:], write_grid):
    for label in policy_names:
        measured = [
            item
            for item in mixed_results
            if item["policy"] == label and item["write_variation"] == write
        ]
        ax.plot(
            [item["adc_bits"] for item in measured],
            [100 * item["top1"] for item in measured],
            marker="o",
            label=label,
        )
    ax.axhline(100 * digital_result["top1"], color="black", linestyle="--", label="Digital")
    ax.set(
        xlabel="ADC bits",
        ylabel=f"Top-1 (%) on {len(targets)} test images",
        title=f"Write variation {write:g}",
        xticks=adc_grid,
        ylim=accuracy_limits,
    )
    ax.grid(axis="y", alpha=0.25)
axes[2].legend(fontsize=8)
plt.show()

### Observed execution

Results for **30 policy/ADC/write configurations** use the checkpoint above, test indices 0..511, batch size 8, seed 135, default scheduling and 64 x 64 arrays. Each mapped layer has its own simulator. The table below shows ADC=8 and write=0.02.

| Policy | Correct / 512 | Top-1 (%) |
| --- | --- | --- |
| Digital checkpoint | 481 | 93.95 |
| Uniform 8-bit | 482 | 94.14 |
| Uniform 4-bit | 476 | 92.97 |
| User assignment: first two convolutions and final head at 8-bit | 483 | 94.34 |
| First convolution and final head at 8-bit | 478 | 93.36 |
| Features at 8-bit, classifier at 4-bit | 481 | 93.95 |

Across all 30 configurations, top-1 ranges from 92.97% to 94.34%.

Edit `layer_bits` and `adc_by_layer` to evaluate your own precision assignment.